In [3]:
import sounddevice as sd
import numpy as np
import scipy.io.wavfile as wavfile
import time

def record_audio(duration=35, sample_rate=16000):
    """Record audio for specified duration"""
    print(f"🎤 Recording for {duration} seconds...")
    print("Get ready...")
    time.sleep(2)
    
    # Countdown
    for i in range(3, 0, -1):
        print(f"Starting in {i}...")
        time.sleep(1)
    
    print("🔴 RECORDING! Speak now!")
    
    # Record audio
    audio = sd.rec(int(duration * sample_rate), 
                   samplerate=sample_rate, 
                   channels=1, 
                   dtype='float32')
    sd.wait()
    
    print("✅ Recording complete!")
    return audio.flatten(), sample_rate

# Record your audio
audio, sr = record_audio(duration=50)  # 45 seconds

# Save to file
audio = np.clip(audio, -1, 1)
audio_int16 = (audio * 32767).astype(np.int16)
wavfile.write('my_audio_recording.wav', sr, audio_int16)
print("✅ Saved to 'my_audio_recording.wav'")


🎤 Recording for 50 seconds...
Get ready...
Starting in 3...
Starting in 2...
Starting in 1...
🔴 RECORDING! Speak now!
✅ Recording complete!
✅ Saved to 'my_audio_recording.wav'


In [4]:
import scipy.io.wavfile as wavfile
from openai import OpenAI
import io
import os
from IPython.display import Audio, display
from dotenv import load_dotenv
from pathlib import Path

env_path = Path.cwd().parent / ".env"
load_dotenv(dotenv_path=env_path)

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Create in-memory WAV file from the recorded audio
wav_path = Path("my_audio_recording.wav")
with open(wav_path, "rb") as f:
    wav_bytes = f.read()

audio_buffer = io.BytesIO(wav_bytes)
audio_buffer.name = "my_audio_recording.wav"  # important for Whisper


# Transcribe with Whisper
print("🤖 Transcribing with Whisper...")
transcript = client.audio.transcriptions.create(
    model="whisper-1",
    file=audio_buffer
)

output_path = Path("transcripts") / "whisper_output_sample1.txt"
output_path.parent.mkdir(exist_ok=True)

with output_path.open("w", encoding="utf-8") as f:
    f.write(transcript.text)


print("\n📝 Transcription:")
print("-" * 40)
print(transcript.text)

🤖 Transcribing with Whisper...

📝 Transcription:
----------------------------------------
Hello everyone. This is a totally serious meeting recording. We are here to discuss very important topics, like why COVID disappears so quickly during working hours and how meetings somehow multiply when deadlines get closer. If you hear a chair moving or a keyboard typing, that's just real life happening. Someone might cough, someone else might forget they're on The goal of this recording is not productivity, but to see how well this transcription actually works. If this sentence is perfectly transcribed, then congratulations, technology wins today.


In [9]:
import jiwer
import json

import jiwer

def calculate_wer(reference, hypothesis):
    transformation = jiwer.Compose([
        jiwer.ToLowerCase(),
        jiwer.RemovePunctuation(),
        jiwer.RemoveMultipleSpaces(),
        jiwer.Strip(),
    ])
    
    reference_clean = transformation(reference)
    hypothesis_clean = transformation(hypothesis)
    
    measures = jiwer.process_words(
        reference_clean,
        hypothesis_clean
    )
    
    ref_words = len(reference_clean.split())
    hyp_words = len(hypothesis_clean.split())
    
    return {
        "wer": measures.wer,
        "substitutions": measures.substitutions,
        "insertions": measures.insertions,
        "deletions": measures.deletions,
        "hits": measures.hits,
        "reference_words": ref_words,
        "hypothesis_words": hyp_words,
        "accuracy": 1 - measures.wer
    }


def evaluate_whisper_accuracy(whisper_text, ground_truth_text):
    """
    Evaluate Whisper transcription accuracy against ground truth.
    
    Parameters:
    - whisper_text: Whisper transcription
    - ground_truth_text: Human-verified ground truth transcription
    
    Returns:
    - Dictionary with evaluation results
    """
    print("\n" + "="*50)
    print("EVALUATING WHISPER ACCURACY")
    print("="*50)
    
    wer_metrics = calculate_wer(ground_truth_text, whisper_text)
    
    print(f"\nWord Error Rate (WER): {wer_metrics['wer']:.4f} ({wer_metrics['wer']*100:.2f}%)")
    print(f"Accuracy: {wer_metrics['accuracy']:.4f} ({wer_metrics['accuracy']*100:.2f}%)")
    print(f"\nError Breakdown:")
    print(f"  Substitutions: {wer_metrics['substitutions']}")
    print(f"  Insertions: {wer_metrics['insertions']}")
    print(f"  Deletions: {wer_metrics['deletions']}")
    print(f"  Correct words: {wer_metrics['hits']}")
    print(f"\nWord Counts:")
    print(f"  Reference words: {wer_metrics['reference_words']}")
    print(f"  Hypothesis words: {wer_metrics['hypothesis_words']}")
    
    return wer_metrics


ground_truth_file_path = Path("transcripts/ground_truth_transcription.txt")
whisper_output_file_path = Path("transcripts/whisper_output_sample1.txt")

with ground_truth_file_path.open("r", encoding="utf-8") as f:
    ground_truth_text = f.read()

with whisper_output_file_path.open("r", encoding="utf-8") as f:
    whisper_result = f.read()

# Calculate WER
if whisper_result and ground_truth_text:
    wer_results = evaluate_whisper_accuracy(
        whisper_result,
        ground_truth_text
    )
    
    # Save WER results
    with open("wer_results.json", 'w') as f:
        json.dump(wer_results, f, indent=2)
    
    print("\n✓ WER evaluation saved!")
else:
    print("⚠ Need both Whisper transcription and ground truth to calculate WER")



EVALUATING WHISPER ACCURACY

Word Error Rate (WER): 0.0978 (9.78%)
Accuracy: 0.9022 (90.22%)

Error Breakdown:
  Substitutions: 4
  Insertions: 0
  Deletions: 5
  Correct words: 83

Word Counts:
  Reference words: 92
  Hypothesis words: 87

✓ WER evaluation saved!


In [10]:
from dataclasses import dataclass

@dataclass
class Pricing:
    whisper_per_min: float = 0.006
    gpt4o_mini_transcribe_per_min: float = 0.003

def estimate_transcription_cost(audio_seconds: float, price_per_minute: float) -> float:
    minutes = audio_seconds / 60.0
    return minutes * price_per_minute

# Example: 30 sec
audio_seconds = 30
pricing = Pricing()

whisper_cost = estimate_transcription_cost(audio_seconds, pricing.whisper_per_min)
mini_cost = estimate_transcription_cost(audio_seconds, pricing.gpt4o_mini_transcribe_per_min)

print(f"Whisper cost for {audio_seconds}s: ${whisper_cost:.6f}")
print(f"4o-mini-transcribe cost for {audio_seconds}s: ${mini_cost:.6f}")


Whisper cost for 30s: $0.003000
4o-mini-transcribe cost for 30s: $0.001500
